# 07 · Practice exams, cramming, and the report figures

Two last analyses (timed practice exams and timing of practice), then every figure in `outputs/figures/`,
the headline metrics, and the aggregate tables for Looker Studio.

In [1]:
import pandas as pd
from so_ell import config
from so_ell.notebook import show, show_sql, describe
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
config.ensure_dirs()
T = config.TABLES_DIR

In [2]:
from IPython.display import display
from so_ell import practice_exams, report
from so_ell.scores import build_windows
scores = pd.read_parquet(config.PROCESSED_DIR / "student_exam_scores.parquet")
windows = build_windows(pd.read_parquet(config.EXAM_ATTEMPTS_PARQUET))

## 1. Timed practice exams
Students who finished a timed practice midterm vs those who started one or did none.

In [3]:
show(practice_exams.practice_midterm_analysis)
practice = practice_exams.practice_midterm_analysis(scores)
display(practice["group"].round(2))
practice["signals"].round(3)

`so_ell.practice_exams.practice_midterm_analysis`
```python
def practice_midterm_analysis(scores: pd.DataFrame) -> dict[str, pd.DataFrame]:
    actual = scores[WINDOW_KEYS + ["exam_start", "score_pct"]]
    markers = _read_events(
        ["time", "anon_student_id", "session_id", "problem_name", "selection", "action", "semester"], ["timedExam"]
    )
    markers["midterm"] = markers["problem_name"].map(parse_midterm)
    markers = markers.loc[markers["midterm"].notna() & markers["problem_name"].map(clean_name).str.contains("practice")]

    starts = markers.loc[markers["action"].eq("start")].groupby(ATTEMPT_KEYS)["time"].min().rename("practice_start")
    ends = markers.loc[markers["action"].eq("finish")].groupby(ATTEMPT_KEYS)["time"].max().rename("practice_end")
    names = markers.groupby(ATTEMPT_KEYS)["problem_name"].agg(lambda s: " | ".join(sorted(set(s)))).rename("practice_name")
    attempts = pd.concat([starts, ends, names], axis=1).reset_index().dropna(subset=["practice_start"])
    attempts = attempts.merge(actual[WINDOW_KEYS + ["exam_start"]], on=WINDOW_KEYS, how="inner")
    attempts = attempts.loc[attempts["practice_start"] < attempts["exam_start"]].copy()
    attempts["finished_before_exam"] = attempts["practice_end"].notna() & (attempts["practice_end"] < attempts["exam_start"])

    started = attempts.groupby(WINDOW_KEYS, as_index=False).agg(
        timed_practice_starts=("practice_name", "nunique"),
        timed_practice_finishes=("finished_before_exam", "sum"),
    )

    # Score finished practice attempts using only the self-grading item types.
    answers = _read_events(
        ["time", "anon_student_id", "session_id", "problem_name", "selection", "action", "input", "semester"],
        ["mChoice", "parsons", "hparsonsAnswer", "unittest"],
    )
    answers = add_event_scores(answers)
    finished = attempts.loc[attempts["finished_before_exam"]]
    labeled = answers.merge(
        finished[ATTEMPT_KEYS + ["practice_start", "practice_end", "practice_name"]],
        on=["semester_raw", "anon_student_id", "session_id"],
        how="inner",
    )
    labeled = labeled.loc[labeled["time"].between(labeled["practice_start"], labeled["practice_end"])]
    item_keys = ATTEMPT_KEYS + ["practice_name", "problem_name"]
    items = (
        labeled.sort_values(item_keys + ["time"])
        .groupby(item_keys, as_index=False)
        .agg(family=("family", "first"), last=("event_score", "last"), best=("event_score", "max"))
    )
    items["item_score"] = np.where(items["family"].eq("mchoice"), items["last"], items["best"])
    attempt_scores = items.groupby(ATTEMPT_KEYS + ["practice_name"], as_index=False).agg(
        practice_items=("problem_name", "nunique"), practice_score_pct=("item_score", "mean")
    )
    attempt_scores["practice_score_pct"] *= 100
    best = attempt_scores.groupby(WINDOW_KEYS, as_index=False).agg(best_practice_score_pct=("practice_score_pct", "max"))

    student = actual.merge(started, on=WINDOW_KEYS, how="left").merge(best, on=WINDOW_KEYS, how="left")
    student[["timed_practice_starts", "timed_practice_finishes"]] = (
        student[["timed_practice_starts", "timed_practice_finishes"]].fillna(0).astype(int)
    )
    student["practice_group"] = np.select(
        [student["timed_practice_starts"].eq(0), student["timed_practice_finishes"].gt(0)],
        ["No timed practice", "Finished timed practice"],
        default="Started only",
    )
    student["cohort"] = student["semester_raw"].astype(str) + "_" + student["midterm"].astype(str)

    group = student.groupby(["midterm", "practice_group"], as_index=False).agg(
        n_student_exams=("anon_student_id", "count"),
        mean_exam_score=("score_pct", "mean"),
        median_exam_score=("score_pct", "median"),
    )
    cohort = student.groupby(["semester_raw", "midterm"], as_index=False).agg(
        n_student_exams=("anon_student_id", "count"),
        timed_practice_rate=("timed_practice_starts", lambda s: 100 * s.gt(0).mean()),
        finished_practice_rate=("timed_practice_finishes", lambda s: 100 * s.gt(0).mean()),
        mean_exam_score=("score_pct", "mean"),
        mean_best_practice_score=("best_practice_score_pct", "mean"),
    )
    pivot = student.pivot_table(index="cohort", columns="practice_group", values="score_pct", aggfunc="mean")
    finish_gap = (
        (pivot["Finished timed practice"] - pivot["No timed practice"]).mean()
        if {"Finished timed practice", "No timed practice"} <= set(pivot.columns)
        else np.nan
    )
    signals = pd.DataFrame(
        [
            {"metric": "practice_score_rank_corr", "value": within_group_rank_corr(student, "cohort", "best_practice_score_pct", "score_pct"),
             "description": "Within-cohort rank correlation, best timed-practice score vs real midterm score."},
            {"metric": "practice_score_quartile_gap", "value": within_group_quartile_gap(student, "cohort", "best_practice_score_pct", "score_pct"),
             "description": "Top minus bottom quartile of practice score, mean real midterm points."},
            {"metric": "finished_vs_none_gap", "value": finish_gap,
             "description": "Mean within-cohort gap: finished a timed practice exam vs did none."},
            {"metric": "finished_practice_attempts", "value": float(len(finished)),
             "description": "Finished timed practice attempts before the real exam."},
        ]
    )
    return {"student": student, "attempts": attempt_scores, "group": group, "cohort": cohort, "signals": signals}
```

,midterm,practice_group,n_student_exams,mean_exam_score,median_exam_score
0,mid1,Finished timed practice,829,74.37,78.60
1,mid1,No timed practice,310,70.07,75.86
2,mid1,Started only,240,70.29,73.28
3,mid2,Finished timed practice,636,66.30,68.62
4,mid2,No timed practice,504,64.68,67.32
5,mid2,Started only,226,62.83,63.48


,metric,value,description
0,practice_score_rank_corr,0.317,"Within-cohort rank correlation, best timed-pra..."
1,practice_score_quartile_gap,11.597,"Top minus bottom quartile of practice score, m..."
2,finished_vs_none_gap,5.175,Mean within-cohort gap: finished a timed pract...
3,finished_practice_attempts,1824.000,Finished timed practice attempts before the re...


## 2. Cramming vs spacing
Share of each student's last-28-day practice that fell in each window before the exam, by score group.

In [4]:
show(practice_exams.cramming_analysis)
cram = practice_exams.cramming_analysis(scores, windows)
cram["summary"].round(3)

`so_ell.practice_exams.cramming_analysis`
```python
def cramming_analysis(scores: pd.DataFrame, windows: pd.DataFrame) -> dict[str, pd.DataFrame]:
    from so_ell.scoring import EXAM_SELECTIONS

    events = _read_events(["time", "anon_student_id", "selection", "semester"], EXAM_SELECTIONS)
    events = drop_exam_window_events(events, windows)
    events["event_day"] = events["time"].dt.floor("D")
    pre = events[["semester_raw", "anon_student_id", "event_day"]].merge(
        scores[WINDOW_KEYS + ["exam_start", "score_pct"]], on=["semester_raw", "anon_student_id"], how="inner"
    )
    pre["days_before"] = (pre["exam_start"].dt.floor("D") - pre["event_day"]).dt.days
    pre = pre.loc[pre["days_before"].between(1, 28)].copy()
    pre["cohort"] = pre["semester_raw"].astype(str) + "_" + pre["midterm"].astype(str)

    band = scores[WINDOW_KEYS + ["score_pct"]].copy()
    band["cohort"] = band["semester_raw"].astype(str) + "_" + band["midterm"].astype(str)
    band["score_group"] = band.groupby("cohort")["score_pct"].transform(
        lambda s: pd.qcut(s.rank(method="first"), 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
    )
    pre = pre.merge(band[WINDOW_KEYS + ["score_group"]], on=WINDOW_KEYS)

    daily = pre.groupby(WINDOW_KEYS + ["score_group", "days_before"], as_index=False, observed=True).size()
    daily["share"] = daily["size"] / daily.groupby(WINDOW_KEYS)["size"].transform("sum")
    metrics = daily.groupby(WINDOW_KEYS + ["score_group"], as_index=False, observed=True).apply(
        lambda g: pd.Series(
            {
                "active_days_28d": g["days_before"].nunique(),
                "last3_share": g.loc[g["days_before"] <= 3, "share"].sum(),
                "last7_share": g.loc[g["days_before"] <= 7, "share"].sum(),
                "concentration_hhi": (g["share"] ** 2).sum(),
            }
        ),
        include_groups=False,
    )
    daily["timing_bin"] = pd.cut(
        daily["days_before"], bins=[0, 3, 7, 14, 28],
        labels=["1-3 days before", "4-7 days before", "8-14 days before", "15-28 days before"],
    )
    # Per student: share of their 28-day practice in each bin (0 when empty), then average by band.
    per_student = (
        daily.groupby(WINDOW_KEYS + ["timing_bin"], observed=False, as_index=False)["share"].sum()
        .merge(band[WINDOW_KEYS + ["score_group"]], on=WINDOW_KEYS)
    )
    per_student = per_student.merge(metrics[WINDOW_KEYS], on=WINDOW_KEYS)  # only students with activity
    bins = (
        per_student.groupby(["score_group", "timing_bin"], observed=True, as_index=False)["share"].mean()
        .rename(columns={"share": "mean_share_pct"})
    )
    bins["mean_share_pct"] *= 100
    summary = metrics.groupby("score_group", observed=True, as_index=False).agg(
        n_student_exams=("anon_student_id", "count"),
        mean_active_days_28d=("active_days_28d", "mean"),
        mean_last3_share=("last3_share", "mean"),
        mean_last7_share=("last7_share", "mean"),
        mean_concentration_hhi=("concentration_hhi", "mean"),
    )
    return {"bins": bins, "metrics": metrics, "summary": summary}
```

,score_group,n_student_exams,mean_active_days_28d,mean_last3_share,mean_last7_share,mean_concentration_hhi
0,Q1 lowest,688,11.593,0.239,0.368,0.206
1,Q2,685,11.860,0.274,0.412,0.196
2,Q3,678,11.448,0.273,0.403,0.192
3,Q4 highest,687,11.934,0.270,0.411,0.195


## 3. Build every figure and table
`report.build_all` writes the practice-exam and cramming tables above, all figures, `outputs/headline_metrics.csv` and the Looker export.

In [5]:
show(report.build_all)
report.build_all()
pd.read_csv(config.OUTPUTS_DIR / "headline_metrics.csv").round(3)

`so_ell.report.build_all`
```python
def build_all() -> None:
    style()
    scores = pd.read_parquet(P / "student_exam_scores.parquet")
    features = pd.read_parquet(P / "student_exam_features.parquet")
    windows = build_windows(pd.read_parquet(config.EXAM_ATTEMPTS_PARQUET))
    corr = pd.read_csv(T / "feature_signals.csv")
    fe = pd.read_csv(T / "parsons_models.csv")
    boot = pd.read_csv(T / "parsons_gap_bootstrap.csv").iloc[0]
    quartiles = pd.read_csv(T / "feature_quartile_scores.csv")
    summary = pd.read_csv(T / "cohort_score_summary.csv")
    comparison = pd.read_csv(T / "at_risk_model_comparison.csv")
    lead = pd.read_csv(T / "at_risk_lead_time.csv")
    cal = pd.read_csv(T / "at_risk_calibration.csv")
    sens = pd.read_csv(T / "key_threshold_sensitivity.csv")
    unscored = pd.read_csv(T / "unscored_items_summary.csv")

    practice = practice_exams.practice_midterm_analysis(scores)
    practice["student"].to_parquet(P / "practice_exam_students.parquet", index=False)
    practice["group"].to_csv(T / "practice_exam_groups.csv", index=False)
    practice["cohort"].to_csv(T / "practice_exam_by_cohort.csv", index=False)
    practice["signals"].to_csv(T / "practice_exam_signals.csv", index=False)
    cram = practice_exams.cramming_analysis(scores, windows)
    cram["metrics"].to_parquet(P / "cramming_student_metrics.parquet", index=False)
    cram["bins"].to_csv(T / "cramming_timing_bins.csv", index=False)
    cram["summary"].to_csv(T / "cramming_summary.csv", index=False)

    funnel = pipeline_funnel()
    funnel.to_csv(T / "pipeline_funnel.csv", index=False)

    fig_pipeline_funnel(funnel)
    fig_cohort_scores(summary)
    fig_score_distributions(scores)
    fig_feature_signals(corr)
    fig_parsons_scatter(features)
    fig_parsons_quartiles(quartiles)
    fig_participation(features).to_csv(T / "participation_and_exposure.csv", index=False)
    fig_effort_efficiency(features).to_csv(T / "parsons_effort_profiles.csv", index=False)
    fig_parsons_baseline(features).to_csv(T / "parsons_gain_by_quartile.csv", index=False)
    fig_cramming(cram["bins"])
    fig_practice_exams(practice["group"])
    fig_scoring_coverage(unscored, scores)
    fig_models(comparison, lead)
    fig_calibration(cal)
    fig_sensitivity(sens)

    headline = headline_metrics(corr, fe, boot, comparison, lead, practice["signals"], funnel)
    headline.to_csv(config.OUTPUTS_DIR / "headline_metrics.csv", index=False)
    export_looker(features, corr, fe, comparison, pd.read_csv(T / "chapter_activity_by_cohort.csv"),
                  pd.read_csv(T / "practice_timing_by_score_band.csv"), quartiles, summary)
```

,metric,value,note
0,raw_events,1.980538e+07,Rows across all raw exports.
1,student_exams,2.745000e+03,Reconstructed student-midterm scores.
2,students,1.397000e+03,Distinct students.
3,parsons_quartile_gap,2.039000e+01,Top vs bottom quartile of Parsons first-try su...
4,parsons_quartile_gap_ci_low,1.877900e+01,"95% cluster-bootstrap CI, lower."
5,parsons_quartile_gap_ci_high,2.212700e+01,"95% cluster-bootstrap CI, upper."
6,parsons_rank_corr,5.700000e-01,Within-cohort rank correlation with midterm sc...
7,parsons_fe_coef_per_10pts,4.868000e+00,"Exam points per +10 Parsons points, cohort FE,..."
8,parsons_controlled_coef_per_10pts,4.960000e+00,"Same, plus effort controls."
9,parsons_mid1_controlled_coef_per_10pts,1.618000e+00,"Midterm 2, controlling for midterm 1 score."


## 4. The figures

In [6]:
from IPython.display import Markdown
# Linked rather than embedded, so the notebook stays small; GitHub renders the relative paths.
display(Markdown("\n\n".join(f"**{p.stem}**\n\n![{p.stem}](../outputs/figures/{p.name})" for p in sorted(config.FIGURES_DIR.glob("*.png")))))

**01_pipeline_funnel**

![01_pipeline_funnel](../outputs/figures/01_pipeline_funnel.png)

**02_cohort_scores**

![02_cohort_scores](../outputs/figures/02_cohort_scores.png)

**03_score_distributions**

![03_score_distributions](../outputs/figures/03_score_distributions.png)

**04_feature_signals**

![04_feature_signals](../outputs/figures/04_feature_signals.png)

**05_parsons_scatter**

![05_parsons_scatter](../outputs/figures/05_parsons_scatter.png)

**06_parsons_quartiles**

![06_parsons_quartiles](../outputs/figures/06_parsons_quartiles.png)

**07_participation_and_exposure**

![07_participation_and_exposure](../outputs/figures/07_participation_and_exposure.png)

**08_effort_vs_efficiency**

![08_effort_vs_efficiency](../outputs/figures/08_effort_vs_efficiency.png)

**09_parsons_gain_paradox**

![09_parsons_gain_paradox](../outputs/figures/09_parsons_gain_paradox.png)

**10_cramming_vs_spacing**

![10_cramming_vs_spacing](../outputs/figures/10_cramming_vs_spacing.png)

**11_practice_exams**

![11_practice_exams](../outputs/figures/11_practice_exams.png)

**12_scoring_coverage**

![12_scoring_coverage](../outputs/figures/12_scoring_coverage.png)

**13_at_risk_models**

![13_at_risk_models](../outputs/figures/13_at_risk_models.png)

**14_calibration**

![14_calibration](../outputs/figures/14_calibration.png)

**15_key_threshold_sensitivity**

![15_key_threshold_sensitivity](../outputs/figures/15_key_threshold_sensitivity.png)